# Notebook 2: Movie Chatbot with Web Search

## Learning Objectives
- Add web search capability to chatbot
- Implement tool-based agentic behavior
- Understand when LLM decides to use tools
- Handle tool execution and results

## What's New:
- **Web Search Tool**: Access current information
- **Agentic Behavior**: LLM decides when to search
- **Tool Binding**: Automatic tool selection
- **SERPER API**: Google search integration

In [1]:
# Install packages (includes search tools)
%pip install langchain langchain-community langgraph python-dotenv openpyxl pandas langchain-openai google-search-results --quiet

  DEPRECATION: Building 'google-search-results' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'google-search-results'. Discussion can be found at https://github.com/pypa/pip/issues/6334
Note: you may need to restart the kernel to use updated packages.


In [22]:
import os
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_community.utilities import GoogleSerperAPIWrapper
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from typing import Literal

from ai_course_utils import load_llm_from_env, load_use_case_config, display_config

print("✓ Imports successful")

✓ Imports successful


In [23]:
display_config()

API CONFIGURATION (.env file)
LLM Provider:    openai
LLM Model:       gpt-4.1-mini
Temperature:     0.0

API Keys Status:
  OpenAI               ✓ Set
  Google               ✗ Not set
  Mistral              ✗ Not set
  Anthropic            ✗ Not set
  Serper (Web Search)  ✓ Set

Data Files:
  Provide file paths as function parameters
  Example: load_use_case_config('your_file.xlsx')


In [24]:
# USER INPUT: Provide file path
use_case_file = "../data/use_case_Movie_Recommendation.xlsx"

use_case_config = load_use_case_config(use_case_file)
system_prompt = use_case_config.get("agent_prompt", "You are a helpful movie assistant with web search capability")

print(f"\n📋 System Prompt Preview:")
print(system_prompt[:200] + "...")

✓ Use case loaded: ../data/use_case_Movie_Recommendation.xlsx
  Components: user, agent_prompt, url

📋 System Prompt Preview:
You are a movie expert whose primary goal is to help users with searching for movies. You are friendly and concise. You only provide factual answers to queries, and do not provide answers that are not...


## Define Web Search Tool

In [25]:
@tool
def search_web(query: str) -> str:
    """
    Search the web for current movie information, reviews, and news.
    
    Use this tool when you need:
    - Current movie information (recent releases, box office)
    - Movie reviews and ratings
    - Actor/director information
    - Film festival news
    - Any information not in your training data
    
    Args:
        query: Search query string
        
    Returns:
        Search results as formatted text
    """
    try:
        search = GoogleSerperAPIWrapper()  # Uses SERPER_API_KEY from .env
        results = search.run(query)
        return results
    except Exception as e:
        return f"Search error: {str(e)}. Please check SERPER_API_KEY in .env file."

# List of available tools
tools = [search_web]

print("✓ Web search tool defined")
print(f"✓ Tools available: {[tool.name for tool in tools]}")

✓ Web search tool defined
✓ Tools available: ['search_web']


## Initialize LLM with Tools

In [26]:
# Load LLM and bind tools
llm = load_llm_from_env()
llm_with_tools = llm.bind_tools(tools)

print("✓ LLM initialized with tool binding")
print("✓ LLM can now decide when to use web search")

🤖 Loading LLM: openai / gpt-4.1-mini
✓ LLM initialized with tool binding
✓ LLM can now decide when to use web search


## Build Agent Graph with Tools

In [27]:
def assistant(state: MessagesState) -> dict:
    """
    Assistant node that can use tools.
    The LLM decides whether to use tools based on the query.
    """
    messages = [SystemMessage(content=system_prompt)] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

def should_continue(state: MessagesState) -> Literal["tools", "__end__"]:
    """
    Determine if we should continue to tools or end.
    """
    last_message = state["messages"][-1]
    
    # If LLM made tool calls, route to tools node
    if last_message.tool_calls:
        return "tools"
    
    # Otherwise, end
    return "__end__"

# Build graph
graph_builder = StateGraph(MessagesState)

# Add nodes
graph_builder.add_node("assistant", assistant)
graph_builder.add_node("tools", ToolNode(tools))

# Add edges
graph_builder.add_edge(START, "assistant")
graph_builder.add_conditional_edges(
    "assistant",
    should_continue,
    ["tools", "__end__"]
)
graph_builder.add_edge("tools", "assistant")

# Compile
memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

print("✓ Agent graph built with tool support")

✓ Agent graph built with tool support


In [28]:
def chat(user_input: str, thread_id: str = "default", verbose: bool = False) -> str:
    """
    Chat with the agent that can use web search.
    
    Args:
        user_input: User's message
        thread_id: Conversation thread
        verbose: Show tool usage
    """
    config = {"configurable": {"thread_id": thread_id}}
    
    if verbose:
        print(f"\n{'='*70}")
        print(f"Processing: {user_input}")
        print(f"{'='*70}")
    
    tools_used = []
    
    for event in graph.stream({"messages": [HumanMessage(content=user_input)]}, config, stream_mode="values"):
        last_message = event["messages"][-1]
        
        # Track tool usage
        if verbose and hasattr(last_message, 'tool_calls') and last_message.tool_calls:
            for tool_call in last_message.tool_calls:
                tool_name = tool_call['name']
                tools_used.append(tool_name)
                print(f"  🔧 Using tool: {tool_name}")
    
    if verbose and tools_used:
        print(f"  ✓ Tools used: {', '.join(tools_used)}")
    elif verbose:
        print(f"  ℹ️ No tools used (answered directly)")
    
    return event["messages"][-1].content

print("🎬 Agent with Web Search Ready!")

🎬 Agent with Web Search Ready!


## Test Examples

In [36]:
# Example 0: Query that should use web search (current info)
print("Example 1: Current Information (Should use search)")
print("="*70)
query1 = "What is the Genre for Toy Story using IMDB.  Is it same for TMDB"
print(f"User: {query1}")
response1 = chat(query1, thread_id="test1", verbose=True)
print(f"\nBot: {response1}")

Example 1: Current Information (Should use search)
User: What is the Genre for Toy Story using IMDB.  Is it same for TMDB

Processing: What is the Genre for Toy Story using IMDB.  Is it same for TMDB
  🔧 Using tool: search_web
  🔧 Using tool: search_web
  ✓ Tools used: search_web, search_web

Bot: The genre of "Toy Story" according to IMDb is Adventure, Animation, Comedy, Family, and Fantasy. The same genres—Family, Comedy, Animation, and Adventure—are also listed on TMDB. So, the genre classification is essentially the same across both IMDb and TMDB. Would you like to know more about "Toy Story" or explore similar movies?


In [33]:
# Example 1: Query that should use web search (current info)
print("Example 1: Current Information (Should use search)")
print("="*70)
query1 = "Provide cast of the movie using url - https://www.imdb.com/title/tt0113041/"
print(f"User: {query1}")
response1 = chat(query1, thread_id="test1", verbose=True)
print(f"\nBot: {response1}")

Example 1: Current Information (Should use search)
User: Provide cast of the movie using url - https://www.imdb.com/title/tt0113041/

Processing: Provide cast of the movie using url - https://www.imdb.com/title/tt0113041/
  🔧 Using tool: search_web
  ✓ Tools used: search_web

Bot: The main cast of "Father of the Bride Part II" includes:
- Steve Martin as George Banks
- Diane Keaton as Nina Banks
- Martin Short as Franck Eggelhoffer
- Kimberly Williams-Paisley as Annie Banks-MacKenzie
- George Newbern

If you want, I can provide more details about the cast or other information about the movie. Would you like that?


In [19]:
# Example 2: Query that might not need search (general knowledge)
print("\nExample 2: General Knowledge (May not need search)")
print("="*70)
query2 = "What genre is The Shawshank Redemption?"
print(f"User: {query2}")
response2 = chat(query2, thread_id="test2", verbose=True)
print(f"\nBot: {response2}")


Example 2: General Knowledge (May not need search)
User: What genre is The Shawshank Redemption?

Processing: What genre is The Shawshank Redemption?
  ℹ️ No tools used (answered directly)

Bot: The Shawshank Redemption is primarily a drama film. It also has elements of crime and thriller genres. Would you like to know more about the movie or perhaps some similar movies in the same genre?


In [20]:
# Example 3: Recent movie information
print("\nExample 3: Recent Movie (Should use search)")
print("="*70)
query3 = "Tell me about the latest Dune movie"
print(f"User: {query3}")
response3 = chat(query3, thread_id="test3", verbose=True)
print(f"\nBot: {response3}")


Example 3: Recent Movie (Should use search)
User: Tell me about the latest Dune movie

Processing: Tell me about the latest Dune movie
  🔧 Using tool: search_web
  ✓ Tools used: search_web

Bot: The latest Dune movie is "Dune: Part Two," which continues the story of Paul Atreides as he unites with Chani and the Fremen on a warpath of revenge against conspirators. It follows the 2021 film "Dune," directed by Denis Villeneuve. Additionally, "Dune: Part Three" is scheduled for release on December 18, 2026. Would you like to know more about the cast, plot details, or reviews of "Dune: Part Two"?


## Interactive Testing

In [21]:
# YOUR TURN: Test with your own queries
my_query = "What are the Oscar nominations for 2026?"

print(f"User: {my_query}")
print(f"Bot: {chat(my_query, thread_id='my_session', verbose=True)}")

User: What are the Oscar nominations for 2026?

Processing: What are the Oscar nominations for 2026?
  🔧 Using tool: search_web
  ✓ Tools used: search_web
Bot: The Oscar nominations for 2026 have not been announced yet. The event is scheduled for March 15, 2026. If you want, I can help you keep track of the nominations as they are announced closer to the date. Would you like that?


## Summary

### What We Added:
✅ Web search capability via SERPER API  
✅ Agentic tool selection (LLM decides when to search)  
✅ Tool binding and execution  
✅ Verbose mode to see tool usage  

### Key Concepts:
- **Tools**: Functions the LLM can call
- **Tool Binding**: Attaching tools to LLM
- **Agentic Behavior**: LLM autonomously decides when to use tools
- **Conditional Edges**: Graph routing based on tool calls

### When Does It Search?
- Current events (box office, releases)
- Recent movies or news
- Specific facts that might have changed
- When explicitly asked to search

### Next Steps:
**Notebook 3** adds URL fetching for accessing specific web pages!

### Experiments:
1. Test queries with different information recency
2. Compare responses with and without search
3. Try asking explicitly "Search for..."
4. Monitor which queries trigger search